# SGEMV

In [1]:
!nvidia-smi

Sat Jan 24 16:24:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile utils.h
/**
 * @file utils.h
 * @brief Utility functions for CUDA development
 *
 * This header provides core utility functions focusing on error checking timing,
 * initialising arrays in memory and result comparison.
 */

#ifndef UTILS_H
#define UTILS_H

#include <iostream>
#include <chrono>
#include <cstdlib>
#include <ctime>
#include <cmath>
#include <cuda_runtime.h>

/**
 * @brief CUDA error checking macro
 *
 * Evaluates a CUDA runtime call and checks for errors.
 * If an error is detected, prints detailed information and terminates the program.
 */
#define CUDA_CHECK(call) \
do { \
    cudaError_t error = call; \
    if (error != cudaSuccess) { \
        std::cerr << "CUDA error at " << __FILE__ << ":" << __LINE__ << " - " \
        << cudaGetErrorString(error) << " (" << error << ") " << std::endl; \
        exit(EXIT_FAILURE); \
} \
}while(0)

/**
 * @brief Initialise multiple arrays with random values in a specified range
 *
 * @param arrays     Array of pointers to initialize
 * @param num_arrays Number of arrays to initialize
 * @param size       Number of elements in each array
 * @param min       Minimum value for random numbers (default: 0.0)
 * @param max       Maximum value for random numbers (default: 1.0)
 * @param seed       Seed for random generator, 0 means use time(0) (default: 0)
 */
void initialiseArrays(float** arrays, int num_arrays, size_t size, float min=0.0f, float max=1.0f, unsigned int seed=0)
{
    // Set random seed
    if (seed == 0)
    {
        seed = static_cast<unsigned int>(time(0)); // get current time
    }
    srand(seed);

    float range = max - min;

    for (int i = 0; i < num_arrays; i++) // Iterate through each array pointer
    {
        for (size_t j = 0; j < size; j++) // Iterate through each element
        {
            arrays[i][j] = min + (static_cast<float>(rand()) / RAND_MAX) * range;
        }
    }
}

/**
 * @brief Measure CPU execution time using std::chrono
 *
 * @tparam Func     Function type
 * @param function  Function or lambda to measure
 * @return double   Execution time in milliseconds
 */
template <typename Function>
double measureExecutionTime(Function function)
{
    auto start = std::chrono::steady_clock::now();
    function();
    auto end = std::chrono::steady_clock::now();
    std::chrono::duration<double, std::milli> duration = (end - start);
    return duration.count();
}

/**
 * @brief Measure GPU kernel execution time using CUDA events
 *
 * @tparam KernelFunc  Kernel function type
 * @tparam Args        Kernel argument types
 * @param kernel       Kernel function to measure
 * @param grid         Grid dimensions
 * @param block        Block dimensions
 * @param args         Kernel arguments
 * @return float       Execution time in milliseconds
 */
template <typename KernelFunc>
float measureKernelTime(KernelFunc kernel)
{
    cudaEvent_t start;
    cudaEvent_t stop;
    float elapsed_time;

    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));

    // Start stopwatch
    CUDA_CHECK(cudaEventRecord(start));
    // Launch kernel
    kernel();
    // Stop stopwatch
    CUDA_CHECK(cudaEventRecord(stop));

    CUDA_CHECK(cudaEventSynchronize(stop));
    CUDA_CHECK(cudaEventElapsedTime(&elapsed_time, start, stop));

    // Free events
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));

    return elapsed_time;
}

/**
 * @brief Compare results between CPU and GPU outputs using absolute and relative tolerance
 *
 * @param cpu_result  CPU computed results
 * @param gpu_result  GPU computed results
 * @param size        Number of elements to compare
 * @param atol        Absolute tolerance (default: 1e-4)
 * @param rtol        Relative tolerance (default: 1e-5)
 * @return bool       True if results match within tolerances, false otherwise
 */
bool compareResults(const float *cpu_result, const float *gpu_result,
                    size_t size, float atol = 1e-4f, float rtol = 1e-5f)
{
    for (size_t i = 0; i < size; i++)
    {
        float a = cpu_result[i];
        float b = gpu_result[i];
        float abs_diff = std::fabs(a - b);
        float rel_diff = abs_diff / std::fmax(std::fabs(a), std::fabs(b));

        if (abs_diff > atol && rel_diff > rtol)
        {
            std::cout << "Mismatch at index " << i
                      << ": CPU = " << a
                      << ", GPU = " << b
                      << ", abs diff = " << abs_diff
                      << ", rel diff = " << rel_diff
                      << std::endl;
            return false;
        }
    }
    return true;
}

#endif


Writing utils.h


In [3]:
%%writefile sgemv.cu
#include <iostream>
#include <stdlib.h>
#include <cuda_runtime.h>
#include <cmath>
#include "utils.h"

#define WARP_SIZE 32

template <const int kWarpSize = WARP_SIZE>
__device__ __forceinline__ float warp_reduce_sum_f32(float val) {
    #pragma unroll
    for (int mask = kWarpSize >>1; mask >= 1; mask >>= 1) {
        val += __shfl_xor_sync(0xffffffff, val, mask);
    }
    return val;
}

/**
grid(M/4), block(32,4) blockDim.x=32=K, blockDim.y=4
// a: MxK, x: Kx1, y: Mx1, compute: y = a * x
*/
template <const int NUM_THREADS=256>
__global__ void sgemv(const float* a, const float* x, float* y, int M, int K) {
    int ty = threadIdx.y;
    int tx = threadIdx.x;
    int block_idx = blockIdx.x;
    int NUM_WARPS_K = (K + WARP_SIZE - 1) / WARP_SIZE;
    
    int lane = tx % WARP_SIZE;
    int m = block_idx * blockDim.y + ty;
    float sum = 0.0f;

    if (m < M) {
        for (int w = 0; w < NUM_WARPS_K; w++) {
            int k = w * WARP_SIZE + lane;
            sum += a[m * K + k] * x[k];
        }
    }

    sum = warp_reduce_sum_f32<WARP_SIZE>(sum);
    if (lane == 0) {
        y[m] = sum;
    }
}

// CPU reference for SGEMV: y = A * x
// A: MxK row-major
// x: length K
// y: length M
void sgemv_cpu_ref(const float* A, const float* x, float* y, int M, int K) {
    for (int m = 0; m < M; ++m) {
        float acc = 0.0f;
        for (int k = 0; k < K; ++k) {
            acc += A[m * K + k] * x[k];
        }
        y[m] = acc;
    }
}

int main() {
    int M = 1 << 10;
    int K = 1 << 10;

    // Calculate sizes
    size_t size_a = M * K * sizeof(float);
    size_t size_x = K * sizeof(float);
    size_t size_y = M * sizeof(float);

    // Create host pointers and allocate host memory
    float* A_host = (float*)malloc(size_a);
    float* x_host = (float*)malloc(size_x);
    float* y_host_cpu = (float*)malloc(size_y);
    float* y_host_gpu = (float*)malloc(size_y);

    // Initialise the arrays 
    float* array_A[] {A_host};
    float* array_x[] {x_host};
    initialiseArrays(array_A, 1, M * K, 0.0f, 1.0f, 0);
    initialiseArrays(array_x, 1, K, 0.0f, 1.0f, 0);

    // Measure CPU reference time
    float cpu_time = measureExecutionTime([&](){
        sgemv_cpu_ref(A_host, x_host, y_host_cpu, M, K);
    });

    // Create device ptrs and allocate device memory
    float* A_device;
    float* x_device; 
    float* y_device;

    CUDA_CHECK(cudaMalloc((void**)&A_device, size_a));
    CUDA_CHECK(cudaMalloc((void**)&x_device, size_x));
    CUDA_CHECK(cudaMalloc((void**)&y_device, size_y));

    // Copy arrays to device memory
    CUDA_CHECK(cudaMemcpy(A_device, A_host, size_a, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(x_device, x_host, size_x, cudaMemcpyHostToDevice));

    // Configure grid and block dims
    dim3 blockDim(32, 4);
    int num_blocks = (M + blockDim.y - 1) / blockDim.y;
    dim3 gridDim(num_blocks);

    // Execute kernel & measure time
    double gpu_time = measureKernelTime([&]() {
        sgemv<<<gridDim, blockDim>>>(A_device, x_device, y_device, M, K);
        CUDA_CHECK(cudaDeviceSynchronize());
    });

    // Copy results back to host
    CUDA_CHECK(cudaMemcpy(y_host_gpu, y_device, size_y, cudaMemcpyDeviceToHost));

    // Compare against cpu reference for correctness
    bool result_match = compareResults(y_host_cpu, y_host_gpu, M, 1e-4, 1e-5);
    std::cout << (result_match ? "Results match!" : "Results do not match!") << std::endl;
    std::cout << "CPU time (ms): " << cpu_time << std::endl;
    std::cout << "GPU time (ms): " << gpu_time << std::endl;

    // Free memories
    free(A_host);
    free(x_host);
    free(y_host_cpu);
    free(y_host_gpu);

    CUDA_CHECK(cudaFree(A_device));
    CUDA_CHECK(cudaFree(x_device));
    CUDA_CHECK(cudaFree(y_device));
    
    return 0;
}


Writing sgemv.cu


In [4]:
!nvcc -O2 -arch=sm_70 sgemv.cu -o sgemv
!./sgemv

Results match!
CPU time (ms): 1.30974
GPU time (ms): 0.1512
